# Tokenization, Hands-On

A focused deep-dive to go with `docs/04-Tokens-and-Tokenization.md`.

Tokens are the real unit of cost, rate limits, and context-window usage
for any LLM call - the same role vCPU-seconds or request units play in
the infra I already operate. This notebook builds that intuition by
counting and inspecting real tokens, using text that actually shows up
in DevOps work (kubectl commands, YAML, log lines) instead of generic
sentences.

**Setup:** `pip install tiktoken`

In [ ]:
import tiktoken

# gpt-4's tokenizer is a solid general-purpose reference, even if you
# call a different model day to day - tokenizers differ slightly
# between providers, but the shape of the lesson holds everywhere.
enc = tiktoken.encoding_for_model("gpt-4")

def show_tokens(text: str):
    ids = enc.encode(text)
    pieces = [enc.decode([i]) for i in ids]
    print(f"text:   {text!r}")
    print(f"tokens: {pieces}")
    print(f"count:  {len(ids)}")

## 1. A token is not a word

Run the same short sentence through the tokenizer and look at how it
actually gets split.

In [ ]:
show_tokens("The pod is stuck in CrashLoopBackOff.")

Notice `CrashLoopBackOff` almost certainly gets split into multiple
pieces - it's a compound, domain-specific term that wasn't common
enough in general training text to earn its own single token. Compare
that to a common word like `"the"`, which is always a single token.

In [ ]:
show_tokens("the")
show_tokens("CrashLoopBackOff")
show_tokens("Kubernetes")

## 2. Technical text costs more tokens per character

This is the part that matters for budgeting and context-window
planning. Compare plain English against a `kubectl` command and a YAML
snippet of similar length.

In [ ]:
samples = {
    "plain_english": "The deployment finished successfully this morning.",
    "kubectl_command": "kubectl apply -f deployment.yaml --namespace production",
    "yaml_snippet": (
        "apiVersion: apps/v1\n"
        "kind: Deployment\n"
        "metadata:\n"
        "  name: web-app\n"
        "spec:\n"
        "  replicas: 3\n"
    ),
}

print(f"{'sample':<18} {'chars':>6} {'tokens':>7} {'tokens/char':>12}")
print("-" * 48)
for name, text in samples.items():
    n_chars = len(text)
    n_tokens = len(enc.encode(text))
    print(f"{name:<18} {n_chars:>6} {n_tokens:>7} {n_tokens / n_chars:>12.2f}")

YAML and shell commands consistently tokenize *worse* (more tokens per
character) than plain English, because of all the punctuation,
indentation, and short identifiers. Worth remembering before pasting a
whole manifest or log file into a prompt "because it looks short."

## 3. Estimate real cost from token counts

Providers price per 1M tokens. Once you can count tokens, you can
estimate cost for a given workload before you run it - the same
instinct as estimating cloud spend before provisioning.

In [ ]:
def estimate_cost(text: str, price_per_million_input=3.00, avg_output_tokens=200, price_per_million_output=15.00):
    input_tokens = len(enc.encode(text))
    input_cost = (input_tokens / 1_000_000) * price_per_million_input
    output_cost = (avg_output_tokens / 1_000_000) * price_per_million_output
    return input_tokens, input_cost + output_cost

log_file_sample = samples["yaml_snippet"] * 50  # simulate a bigger file
tokens, cost = estimate_cost(log_file_sample)
print(f"input tokens: {tokens}")
print(f"estimated cost for one call: ${cost:.4f}")
print(f"estimated cost for 10,000 calls/month: ${cost * 10_000:.2f}")

## Takeaways

- A token is a subword chunk, not a word - common words are one
  token, technical/compound terms often split into several.
- YAML, shell commands, and log lines tokenize worse than plain
  English per character - budget accordingly.
- Token counts are the direct input to both **cost** estimates and
  **context window** checks (`docs/09-Context-Window.md`) - counting
  them before you send a request beats discovering the limit via a
  failed call.

Next: `LLM_Fundamentals.ipynb` for the broader hands-on tour.